## Calibration Design

In [ ]:
def residence_time(seep_k):
    """
    Returnes the residence time (in days) for a given seepage constant (seep_k) (in 1/d units).
    """
    storage = 1
    residence_time = 0
    while storage > 0.5:
        storage *= (1 - seep_k)
        residence_time += 1
    return residence_time

In [ ]:
residence_time(0.005)

# Generate watershed characteristics for user review

In [ ]:
import geopandas as gpd
import rasterio

In [ ]:
watershed_stats = dict(wsc_num = None, 
                       total_area_km2 = None,
                       lu_lc_stats = None,
                       deps_gt_10cm_km2 = None,
                       deps_gt_10cm_pct = None,
                       total_storage_m3 = None,
                       phydap_cover_pct = None,
                       canada_pct = None)

In [ ]:
watershed_stats['wsc_num'] = '05MH008' # WSC number is provided manually by the user, but can also be found in the name of the .rvt file

watershed_shapefile = gpd.read_file("../Data/Cypress_New/DepressionDepths/Raw/finalcat_info_v1-0.shp") # This is the CLRH shapefile used in Depressions depths analysis and Raven file generation
watershed_stats['total_area_km2'] = watershed_shapefile['BasArea'].sum()*10**(-6)




In [ ]:
lulc_raster = rasterio.open("../Data/supporting_GIS/landcover2020_reclass_pra.tif")



In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import pandas as pd

def calculate_class_areas(raster_src, watershed_shapefile, pixel_size=None, crs_match=True):
    """
    Calculate total area for each raster class within a watershed polygon.
    
    Parameters:
    -----------
    raster_src : rasterio dataset
        The LULC raster dataset
    watershed_shapefile : str or gpd.GeoDataFrame
        Path to watershed shapefile or GeoDataFrame
    pixel_size : float, optional
        Pixel size in ground units (m²). If None, calculated from raster metadata.
    crs_match : bool
        If True, reprojects shapefile to match raster CRS
        
    Returns:
    --------
    pd.DataFrame with columns: class, pixel_count, area_m2, area_km2
    """
    
    # Load watershed shapefile if string path provided
    if isinstance(watershed_shapefile, str):
        watershed_gdf = gpd.read_file(watershed_shapefile)
    else:
        watershed_gdf = watershed_shapefile
    
    # Reproject to match raster CRS if needed
    if crs_match and watershed_gdf.crs != raster_src.crs:
        watershed_gdf = watershed_gdf.to_crs(raster_src.crs)
    
    # Get geometry as GeoJSON for masking
    geoms = watershed_gdf.geometry.values
    
    # Mask raster to watershed boundary
    masked_raster, masked_transform = mask(raster_src, geoms, crop=True)
    
    # Use first band if multi-band
    if masked_raster.ndim == 3:
        masked_data = masked_raster[0]
    else:
        masked_data = masked_raster
    
    # Get pixel size if not provided
    if pixel_size is None:
        # Pixel size from raster resolution
        pixel_size = abs(raster_src.res[0] * raster_src.res[1])  # in m²
    
    # Calculate areas for each class
    unique_classes, pixel_counts = np.unique(masked_data, return_counts=True)
    
    # Build results dataframe
    results = []
    for class_id, count in zip(unique_classes, pixel_counts):
        # Skip nodata values (typically 0 or 255, adjust if needed)
        if count > 0:  # Keep all non-zero counts
            area_m2 = count * pixel_size
            area_km2 = area_m2 / 1_000_000
            results.append({
                'class': int(class_id),
                'pixel_count': int(count),
                'area_m2': area_m2,
                'area_km2': area_km2
            })
    
    results_df = pd.DataFrame(results)
    results_df = results_df[results_df['class'] != 0]  # Filter out nodata class if needed
    return results_df.sort_values('class').reset_index(drop=True)

# Usage example:
# watershed_path = r'path\to\your\watershed.shp'  # Update path
# class_areas = calculate_class_areas(lulc_raster, watershed_path)
# print(class_areas)

In [ ]:
watershed_stats['lu_lc_stats'] = calculate_class_areas(lulc_raster, watershed_shapefile)

watershed_stats['lu_lc_stats']['pct'] = (watershed_stats['lu_lc_stats']['area_km2'] / watershed_stats['total_area_km2']) * 100

In [ ]:
depressions_raster = rasterio.open("../Data/Cypress_New/DepressionDepths/Processed/Pembina_LiDAR_DEM_clip_resample_5m_FillBurn_Deps_Corr.tif")

In [ ]:
# Read depressions raster data
depression_data = depressions_raster.read(1)  # Read first band
nodata_value = depressions_raster.nodata

# Create a copy and mask values below 0.1
depression_masked = depression_data.copy().astype('float32')
depression_masked[depression_data < 0.1] = np.nan  # Set to NaN for no-data

# Get pixel resolution (assuming square pixels)
pixel_res = depressions_raster.res
pixel_area_m2 = abs(pixel_res[0] * pixel_res[1])
pixel_area_km2 = pixel_area_m2 / 1_000_000

# Calculate total area (count non-null pixels)
valid_pixels = ~np.isnan(depression_masked)
total_valid_pixels = np.sum(valid_pixels)
total_area_km2 = total_valid_pixels * pixel_area_km2

# Calculate total volume (sum of depth values times pixel area)
total_volume_m3 = np.nansum(depression_masked) * pixel_area_m2
total_volume_km3 = total_volume_m3 / 1e9  # Convert to km³

watershed_stats['deps_gt_10cm_km2'] = total_area_km2
watershed_stats['deps_gt_10cm_pct'] = (total_area_km2 / watershed_stats['total_area_km2']) * 100
watershed_stats['total_storage_m3'] = total_volume_m3

# Display results
print(f"Depression Analysis Results:")
print(f"{'='*50}")
print(f"Pixel size: {pixel_res[0]:.2f} x {abs(pixel_res[1]):.2f} m")
print(f"Pixel area: {pixel_area_m2:.2f} m²")
print(f"\nNumber of valid pixels (depth >= 0.1): {total_valid_pixels}")
print(f"Total area (non-null): {total_area_km2:.4f} km²")
print(f"\nTotal volume representation:")
print(f"  {total_volume_m3:,.0f} m³")
print(f"  {total_volume_km3:.6f} km³")

In [ ]:
watershed_stats['lu_lc_stats']